<a href="https://colab.research.google.com/github/tsubasa-iino/psi4book/blob/main/compchem_book_ch03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 3章 Pythonで量子化学計算をはじめよう

### 環境構築

#### Google Colab上にPsi4をインストール

In [1]:
!pip install -q condacolab
import condacolab
import os

# バグ回避パッチ
if "LD_LIBRARY_PATH" not in os.environ:
    os.environ["LD_LIBRARY_PATH"] = ""

print("Installing CondaColab (Base)...")
condacolab.install()

Installing CondaColab (Base)...
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


ランタイム｜セッションを再起動する｜はい を実行。

In [ ]:
import condacolab
import os
import sys

condacolab.check()

# 1. 邪魔なPinningを削除
if os.path.exists("/usr/local/conda-meta/pinned"):
    !rm /usr/local/conda-meta/pinned

# 2. Python 3.12 と Psi4 をインストール（ディスク書き換え）
print("Upgrading Python to 3.12 & Installing Psi4...")
!mamba install -y -q python=3.12 psi4 -c conda-forge/label/libint_dev -c conda-forge

# 3. Pinningの復元（成功環境の再現）
os.makedirs("/usr/local/conda-meta", exist_ok=True)
with open("/usr/local/conda-meta/pinned", "w") as f:
    f.write("python 3.12.*\n")

# 4. 【最重要】カーネルの自殺（強制再起動）
# これにより、メモリ上のPython 3.11を殺し、ディスク上のPython 3.12をロードさせます
print("\n🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...")
import time
time.sleep(1)
os.kill(os.getpid(), 9)

✨🍰✨ Everything looks OK!
Upgrading Python to 3.12 & Installing Psi4...
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done

🔄 RESTARTING KERNEL TO LOAD PYTHON 3.12...


ランタイム｜セッションを再起動する｜はい を実行。

In [1]:
import sys
import os

# パスが通っていなければ通す
target_path = "/usr/local/lib/python3.12/site-packages"
if target_path not in sys.path:
    sys.path.insert(0, target_path)

import psi4
print(f"✅ Restart Successful.")
print(f"Psi4 Version: {psi4.__version__}")
print(f"Python Version: {sys.version.split()[0]}") # ここが3.12になっているはず

# 計算テスト
psi4.set_memory('500 MB')
mol = psi4.geometry("O\nH 1 0.96\nH 1 0.96 2 104.5")
en = psi4.energy('scf/cc-pvdz')
print(f"Energy: {en:.6f}")

✅ Restart Successful.
Psi4 Version: 1.10
Python Version: 3.12.12
Energy: -76.026633


ここまででインストール確認完了。

In [2]:
import os
import datetime
import numpy as np
import pandas as pd
import psi4

print(f'current time: {datetime.datetime.now()}')
print(f'python version:\n{sys.version}')
print(f'numpy version: {np.__version__}')
print(f'pandas version: {pd.__version__}')
print(f'psi4 version: {psi4.__version__}')

current time: 2026-01-12 07:34:18.550132
python version:
3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy version: 2.0.2
pandas version: 2.2.2
psi4 version: 1.10


#### 計算資源の設定

In [3]:
# 計算資源の確認（CPU, RAM）
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
address sizes

In [4]:
!cat /proc/meminfo

MemTotal:       13286964 kB
MemFree:          701204 kB
MemAvailable:   11895924 kB
Buffers:          174700 kB
Cached:         10845632 kB
SwapCached:            0 kB
Active:          1163756 kB
Inactive:       10718408 kB
Active(anon):       2632 kB
Inactive(anon):   862456 kB
Active(file):    1161124 kB
Inactive(file):  9855952 kB
Unevictable:           8 kB
Mlocked:               8 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:              4856 kB
Writeback:            84 kB
AnonPages:        861700 kB
Mapped:           546352 kB
Shmem:              3248 kB
KReclaimable:     503288 kB
Slab:             576228 kB
SReclaimable:     503288 kB
SUnreclaim:        72940 kB
KernelStack:        5908 kB
PageTables:        14876 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     6643480 kB
Committed_AS:    3080912 kB
VmallocTotal:   34359738367 kB
VmallocUsed:       12392 kB
VmallocChunk:    

In [5]:
n_cpu = os.cpu_count()

In [6]:
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)

In [7]:
# 環境に応じて計算資源を設定
psi4.set_num_threads(n_cpu)
psi4.set_memory(f'{ram * 0.9: .0f}GB')

11000000000

### 3.3 水分子のエネルギー計算をしてみよう

#### 3.3.1 Psithon形式で計算を実行してみよう

In [8]:
!which psi4

/usr/local/bin/psi4


In [10]:
# h2o_psithon.inファイルの作成
%%writefile h2o_psithon.in
molecule h2o {
0 1
O  -0.1176269719      0.7387773605      0.0000000000
H   0.8523730281      0.7387773605      0.0000000000
H  -0.4409567836      1.4770262439     -0.5397651517
}

energy('hf/sto-3g')

Overwriting h2o_psithon.in


上記は書籍から記述変更。
%%writefile h2o_psithon.in
で直接ファイル生成が可能。

In [11]:
!ls -l

total 36
-rw-r--r-- 1 root root 21477 Jan 12 07:32 condacolab_install.log
-rw-r--r-- 1 root root   201 Jan 12 07:35 h2o_psithon.in
-rw-r--r-- 1 root root    17 Jan 12 07:34 psi.1370.clean
drwxr-xr-x 1 root root  4096 Dec 11 14:34 sample_data


In [12]:
!cat h2o_psithon.in

molecule h2o {
0 1
O  -0.1176269719      0.7387773605      0.0000000000
H   0.8523730281      0.7387773605      0.0000000000
H  -0.4409567836      1.4770262439     -0.5397651517
}

energy('hf/sto-3g')


In [13]:
# 計算の実行
!psi4 -i h2o_psithon.in -o h2o_psithon.out

In [14]:
!grep 'Total Energy' h2o_psithon.out

                           Total Energy        Delta E     RMS |[F,P]|
    Total Energy =                        -74.9617284312868719


#### 3.3.2　PsiAPI形式での計算を実行してみよう

In [15]:
# 計算ログファイルの設定
psi4.set_output_file('h2o_psiapi.log')

PosixPath('h2o_psiapi.log')

In [16]:
# 水分子の構造定義
h2o = psi4.geometry('''
0 1
O       -0.1176269719      0.7387773605      0.0000000000
H        0.8523730281      0.7387773605      0.0000000000
H       -0.4409567836      1.4770262439     -0.5397651517
''')

In [17]:
# エネルギー計算の実行
psi4.energy('hf/sto-3g', molecule=h2o)

-74.96172843128687